In [ ]:
# import
import pandas as pd
import requests
from IPython.display import display
from datetime import datetime

In [ ]:
# ETL(Extract, Transform, Load) Block:
def get_flight_data():
    # API - מאגר טיסות - EXTRACT
   
    # מזהה משאב(מגיע מהאתר)
    resource_id = "e83f763b-b7d7-479e-b172-ae981ddc6de5"

    url = f"https://data.gov.il/api/3/action/datastore_search?resource_id={resource_id}"

    # send requests to website and get response(code)
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"There is a problem: {response.status_code}")
        return None
    
    # parsing  the data we got
    data = response.json()
    
    # convert to DataFrame
    records = data['result']['records']
    df_flights = pd.DataFrame(records)
    
    #TRANSFORM

    # Rename columns to meaningful names
    df_flights = df_flights.rename(columns={
        '_id': 'id',
        'CHOPER': 'airline_code',
        'CHOPERD': 'airline_name',
        'CHFLTN': 'flight_number',
        'CHAORD': 'direction',
        'CHSTOL': 'scheduled_time',
        'CHPTOL': 'actual_time',
        'CHLOC1': 'airport_code',
        'CHLOC1D': 'airport_name',
        'CHLOC1TH': 'city_hebrew',
        'CHLOC1T': 'city_english',
        'CHLOC1CH': 'country_hebrew',
        'CHLOCCT': 'country_english',
        'CHTERM': 'terminal',
        'CHCINT': 'checkin_counters',
        'CHCKZN': 'checkin_zone',
        'CHRMINE': 'flight_status',
        'CHRMINH': 'flight_status_hebrew'
    }, errors='ignore')

    df_flights['terminal'] = pd.to_numeric(df_flights['terminal'], errors='coerce').astype('Int8')
    df_flights['flight_number'] = df_flights['flight_number'].astype(str)
    df_flights['scheduled_time'] = pd.to_datetime(df_flights['scheduled_time'])
    df_flights['actual_time'] = pd.to_datetime(df_flights['actual_time'])
    df_flights['terminal'] = pd.to_numeric(df_flights['terminal'], errors='coerce').astype('Int8')

    # missing values:
    # flights that are landing they are arrivel = mean no need to checkin so fill the empty to 'N/A'
    df_flights.loc[df_flights['direction'] == 'A', ['checkin_counters', 'checkin_zone']] = 'N/A'

    # flights that are leaving they are departure = means they do need to do check in so, if there is empty for any reason fill it as 'Missing'
    mask_missing_departures = (df_flights['direction'] == 'D') & (df_flights['checkin_counters'].isna())
    df_flights.loc[mask_missing_departures, ['checkin_counters', 'checkin_zone']] = 'Missing'
    #return to us the df - LOAD
    return df_flights


In [ ]:
# update data
df_flights = get_flight_data()

# quick check for missing and duplicates rows:
# print(f"For df_flights there is: {df_flights.duplicated().sum()} duplicates rows")
# print(f"For df_flights there is: {df_flights.isna().sum()} missing values")


### EDA - Exploratory Data Analysis

In [14]:
# display the table
display(df_flights.head())

# look at the data in static way
df_flights.describe()

,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew,delay
0,1,BZ,755,BLUE BIRD AIRWAYS,2026-06-20 12:15:00,2026-06-20 12:17:00,A,HER,HERAKLION,הרקליון,HERAKLION,יוון,GREECE,3,N/A,N/A,LANDED,נחתה,2.0
1,2,BZ,561,BLUE BIRD AIRWAYS,2026-06-20 13:00:00,2026-06-20 12:34:00,A,BOJ,BURGAS,בורגס,BOURGAS,בולגריה,BULGARIA,3,N/A,N/A,LANDED,נחתה,-26.0
2,3,J2,022,AZERBAIJAN AIRLINES,2026-06-20 12:30:00,2026-06-20 12:38:00,D,GYD,BAKU HEYDAR ALIYEV INT`L,באקו,BAKU,אזרביג'אן,AZERBAIJAN,3,66-70,C,DEPARTED,המריאה,8.0
3,4,LY,9951,EL AL ISRAEL AIRLINES,2026-06-20 12:30:00,2026-06-20 12:38:00,D,GYD,BAKU HEYDAR ALIYEV INT`L,באקו,BAKU,אזרביג'אן,AZERBAIJAN,3,66-70,C,DEPARTED,המריאה,8.0
4,5,EK,2363,EMIRATES,2026-06-20 12:20:00,2026-06-20 12:39:00,A,DXB,DUBAI,דובאי,DUBAI,איחוד האמירויות הערב,UNITED ARAB EMIRATES,3,N/A,N/A,LANDED,נחתה,19.0


,id,scheduled_time,actual_time,terminal,delay
count,2393.00000,2393,2393,2393.0,2393.000000
mean,1197.00000,2026-06-22 14:45:21.186794752,2026-06-22 14:49:38.161303808,3.0,4.282908
min,1.00000,2026-06-20 11:00:00,2026-06-20 12:17:00,3.0,-261.000000
25%,599.00000,2026-06-21 16:40:00,2026-06-21 16:35:00,3.0,0.000000
50%,1197.00000,2026-06-22 14:30:00,2026-06-22 14:30:00,3.0,0.000000
75%,1795.00000,2026-06-23 13:35:00,2026-06-23 13:35:00,3.0,0.000000
max,2393.00000,2026-06-24 12:15:00,2026-06-24 12:15:00,3.0,394.000000
std,690.94392,NaN,NaN,0.0,25.015158


In [ ]:
# create delay column:
df_flights['delay'] = (df_flights['actual_time'] - df_flights['scheduled_time']).dt.total_seconds()/60

display(df_flights.head())

### The Goal of Project:
#### 1.How many flight from start of the day till now?
#### 2. 

In [16]:
# 1.How many flight from start of the day till now?
# to get the date of today
now = datetime.now()
today = now.date()

today_flight = df_flights[
    (df_flights['actual_time'].dt.date == today) &
    (df_flights['actual_time'] <= now)
]
print(f"Today is {today} was {len(today_flight)} flights")
display(today_flight)

Today is 2026-06-21 was 347 flights


,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew,delay
138,139,6H,586,ISRAIR AIRLINES,2026-06-21 00:10:00,2026-06-21 00:01:00,A,LCA,LARNACA,לרנקה,LARNACA,קפריסין,CYPRUS,3,N/A,N/A,LANDED,נחתה,-9.0
139,140,6H,893,ISRAIR AIRLINES,2026-06-20 23:00:00,2026-06-21 00:02:00,D,TBS,TBILISI,טביליסי,TBILISI,גיאורגיה,GEORGIA,3,201-205,G,DEPARTED,המריאה,62.0
140,141,LY,091,EL AL ISRAEL AIRLINES,2026-06-20 23:30:00,2026-06-21 00:04:00,D,NRT,TOKYO INTL.-NARITA,טוקיו,TOKYO,יפן,JAPAN,3,78-99,D,DEPARTED,המריאה,34.0
141,142,QF,5014,QANTAS AIRWAYS,2026-06-20 23:30:00,2026-06-21 00:04:00,D,NRT,TOKYO INTL.-NARITA,טוקיו,TOKYO,יפן,JAPAN,3,78-99,D,DEPARTED,המריאה,34.0
142,143,BZ,611,BLUE BIRD AIRWAYS,2026-06-21 00:20:00,2026-06-21 00:14:00,A,AMS,AMSTERDAM,אמסטרדם,AMSTERDAM,הולנד,NETHERLANDS,3,N/A,N/A,LANDED,נחתה,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,481,AM,7892,AERO MEXICO,2026-06-21 13:55:00,2026-06-21 13:55:00,D,CDG,PARIS-CHARLES DE GAULLE,פריס,PARIS,צרפת,FRANCE,3,78-99,D,ON TIME,בזמן,0.0
481,482,IZ,416,ARKIA ISRAELI AIRLINES,2026-06-21 13:55:00,2026-06-21 13:55:00,A,BUS,BATUMI,בטומי,BATUMI,גיאורגיה,GEORGIA,3,N/A,N/A,NOT FINAL,לא סופי,0.0
482,483,LY,327,EL AL ISRAEL AIRLINES,2026-06-21 13:55:00,2026-06-21 13:55:00,D,CDG,PARIS-CHARLES DE GAULLE,פריס,PARIS,צרפת,FRANCE,3,78-99,D,ON TIME,בזמן,0.0
483,484,SK,3159,SAS,2026-06-21 13:55:00,2026-06-21 13:55:00,D,CDG,PARIS-CHARLES DE GAULLE,פריס,PARIS,צרפת,FRANCE,3,78-99,D,ON TIME,בזמן,0.0


In [ ]:
# 2. Which flight company is popular
flight_company = today_flight['airline_name'].value_counts()
print(flight_company)


airline_name
EL AL ISRAEL AIRLINES             88
ARKIA  ISRAELI  AIRLINES          42
ISRAIR AIRLINES                   38
BLUE BIRD AIRWAYS                 23
AEGEAN  AIRLINES                  14
SAS                               13
AERO MEXICO                       13
TUS AIRWAYS                       11
DELTA AIRLINES                    10
WIZZAIR                            8
EMIRATES                           6
AIR FRANCE                         6
ETIHAD AIRWAYS                     5
FLYDUBAI                           5
TAROM - ROMANIAN AIR TRANSPORT     4
ETHIOPIAN AIRLINES                 4
THAI AIRWAYS                       4
LOT POLISH AIRLINES                4
WIZZ AIR MALTA                     3
AZERBAIJAN AIRLINES                3
QANTAS AIRWAYS                     3
AEROLINEAS ARGENTINAS S.A.         3
AIR PORTUGAL                       3
SKY EXPRESS S.A.                   3
VIETNAM AIRLINES                   3
BELAVIA AIRLINE                    2
FLYONE ARMENIA LLC       